<a href="https://colab.research.google.com/github/pranavib433-glitch/Machine-Learning-Practical/blob/main/Prediction_on_New_DataPoint.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [8]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    classification_report,
    roc_auc_score
)

# Load dataset
df = pd.read_csv("/content/placement_predict_50k Dataset.csv")

# Target
y = df["PlacementStatus"]

# Features
features = [
    "Gender", "City", "CollegeTier", "Stream", "Specialisation",
    "Hostel", "HistoryOfBacklogs",
    "SGPA_Sem1", "SGPA_Sem2", "SGPA_Sem3", "SGPA_Sem4",
    "SGPA_Sem5", "SGPA_Sem6", "SGPA_Sem7", "SGPA_Sem8",
    "CGPA", "AttendancePercent",
    "Internships", "Projects", "Workshops", "Certifications",
    "Publications", "AptitudeTestScore", "SoftSkillsRating",
    "CodingTestScore", "MockInterviewScore", "ExtraCurricular"
]

X = df[features].copy()

# Categorical columns
categorical_features = [
    "Gender", "City", "CollegeTier", "Stream",
    "Specialisation", "Hostel", "HistoryOfBacklogs"
]

# Numerical columns
numerical_features = [
    "SGPA_Sem1", "SGPA_Sem2", "SGPA_Sem3", "SGPA_Sem4",
    "SGPA_Sem5", "SGPA_Sem6", "SGPA_Sem7", "SGPA_Sem8",
    "CGPA", "AttendancePercent",
    "Internships", "Projects", "Workshops", "Certifications",
    "Publications", "AptitudeTestScore", "SoftSkillsRating",
    "CodingTestScore", "MockInterviewScore", "ExtraCurricular"
]

# Missing values
for col in numerical_features:
    X[col] = X[col].fillna(X[col].median())

for col in categorical_features:
    X[col] = X[col].fillna(X[col].mode()[0])

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

# Preprocessing
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numerical_features),
        (
            "cat",
            OneHotEncoder(
                handle_unknown="ignore",
                drop="first"
            ),
            categorical_features
        )
    ]
)

# L2 Logistic Regression
l2_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "classifier",
            LogisticRegression(
                penalty="l2",
                solver="lbfgs",
                C=1.0,
                max_iter=2000,
                random_state=42
            )
        )
    ]
)

# TRAIN
l2_model.fit(X_train, y_train)

# Test prediction
y_pred_l2 = l2_model.predict(X_test)
y_prob_l2 = l2_model.predict_proba(X_test)[:, 1]

# Evaluation
print("============================================")
print("BINOMIAL LOGISTIC REGRESSION RESULTS")
print("============================================")

print("\nAccuracy:", round(
    accuracy_score(y_test, y_pred_l2), 4
))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_l2))

print("\nClassification Report:")
print(
    classification_report(
        y_test,
        y_pred_l2,
        target_names=["Not Placed", "Placed"]
    )
)

print("\nROC-AUC:", round(
    roc_auc_score(y_test, y_prob_l2), 4
))

BINOMIAL LOGISTIC REGRESSION RESULTS

Accuracy: 0.9197

Confusion Matrix:
[[2926  503]
 [ 300 6271]]

Classification Report:
              precision    recall  f1-score   support

  Not Placed       0.91      0.85      0.88      3429
      Placed       0.93      0.95      0.94      6571

    accuracy                           0.92     10000
   macro avg       0.92      0.90      0.91     10000
weighted avg       0.92      0.92      0.92     10000


ROC-AUC: 0.9792
